In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

# load and sort
df = pd.read_csv("data/E2.csv")
matches = df[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]].copy()
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst=True)
matches = matches.sort_values("Date").reset_index(drop=True)

# split
split_idx = int(len(matches) * 0.70)
train = matches.iloc[:split_idx].copy()
test = matches.iloc[split_idx:].copy()

# strengths from train only
avg_home_goals = train["FTHG"].mean()
avg_away_goals = train["FTAG"].mean()
home = train.groupby("HomeTeam").agg(home_scored=("FTHG","mean"), home_conceded=("FTAG","mean"))
away = train.groupby("AwayTeam").agg(away_scored=("FTAG","mean"), away_conceded=("FTHG","mean"))
strength = home.join(away)

def predict_match(home_team, away_team):
    home_exp = strength.loc[home_team,"home_scored"] * strength.loc[away_team,"away_conceded"] / avg_home_goals
    away_exp = strength.loc[away_team,"away_scored"] * strength.loc[home_team,"home_conceded"] / avg_away_goals
    hw, dr, aw = 0.0, 0.0, 0.0
    for hg in range(10):
        for ag in range(10):
            p = poisson.pmf(hg, home_exp) * poisson.pmf(ag, away_exp)
            if hg > ag: hw += p
            elif hg == ag: dr += p
            else: aw += p
    return hw, dr, aw

# predict every test match, store probabilities + actual outcome
rows = []
for _, r in test.iterrows():
    h, a = r["HomeTeam"], r["AwayTeam"]
    if h not in strength.index or a not in strength.index:
        continue
    hw, dr, aw = predict_match(h, a)
    actual = "H" if r["FTHG"] > r["FTAG"] else ("A" if r["FTHG"] < r["FTAG"] else "D")
    rows.append({"p_home": hw, "p_draw": dr, "p_away": aw, "actual": actual})

results = pd.DataFrame(rows)
print(f"Rebuilt {len(results)} test predictions")
results.head()

Rebuilt 166 test predictions


,p_home,p_draw,p_away,actual
0,0.206536,0.237319,0.556132,A
1,0.389746,0.283105,0.327149,A
2,0.482503,0.232681,0.284798,H
3,0.229207,0.267399,0.503391,A
4,0.557849,0.208693,0.233386,H


In [3]:
# mark whether each match was actually a home win (1) or not (0)
results["home_won"] = (results["actual"] == "H").astype(int)

# sort predictions into probability buckets
bins = [0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]
results["bucket"] = pd.cut(results["p_home"], bins)

# for each bucket: how many matches, avg predicted prob, actual home-win rate
calib = results.groupby("bucket", observed=True).agg(
    n=("home_won", "size"),
    avg_predicted=("p_home", "mean"),
    actual_rate=("home_won", "mean")
)
calib

,n,avg_predicted,actual_rate
bucket,,,
"(0.0, 0.2]",15,0.156586,0.266667
"(0.2, 0.3]",35,0.245731,0.114286
"(0.3, 0.4]",24,0.363174,0.500000
"(0.4, 0.5]",37,0.445744,0.405405
"(0.5, 0.6]",30,0.544834,0.533333
"(0.6, 0.7]",19,0.644287,0.631579
"(0.7, 1.0]",6,0.770941,0.666667
